# Counterfactual line structure

Permute ordinary token identities within lines while preserving newline positions.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
rng=np.random.default_rng(6005); counter=tokens.copy()
for start,end in line_segments(tokens,vocab):
 x=counter[start:end].copy(); rng.shuffle(x); counter[start:end]=x
cb=batch.tokens.clone(); cb[0]=torch.as_tensor(counter,device=device)
names=["blocks.1.attn.hook_z","blocks.1.hook_resid_post"]
_,a=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=names); _,b=model.run_with_cache(cb,prepend_bos=False,names_filter=names)
z0,z1=a[names[0]][0,:,1].cpu().numpy(),b[names[0]][0,:,1].cpu().numpy(); p0,p1=a[names[1]][0].cpu().numpy(),b[names[1]][0].cpu().numpy()
print("same newline positions",newline_positions(tokens,vocab)==newline_positions(counter,vocab)); print("head-1 MSE",float(np.mean((z0-z1)**2))); print("post-layer-1 MSE",float(np.mean((p0-p1)**2)))

same newline positions True
head-1 MSE 2.2609939575195312
post-layer-1 MSE 25.77724838256836
